In [13]:
import os
import pandas as pd
import numpy as np


def load_and_combine_csvs(files, period_name=None):
    """
    Load multiple CSV files and combine them into one DataFrame.

    Parameters
    ----------
    files : list of str
        CSV file paths.
    period_name : str, optional
        Label for the period, added as a column.

    Returns
    -------
    pd.DataFrame
    """
    dfs = []

    for file in files:
        df = pd.read_csv(file, low_memory=False, encoding="latin1")

        filename = os.path.basename(file)
        year = filename[:4]

        df["year"] = pd.to_numeric(year, errors="coerce")

        if period_name is not None:
            df["period"] = period_name

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def clean_flight_data(df):
    """
    Clean flight dataset and convert relevant columns to numeric.
    """
    df = df.copy()
    df = df.replace("NA", np.nan)

    numeric_cols = [
        "AirTime",
        "ActualElapsedTime",
        "CRSElapsedTime",
        "ArrDelay",
        "DepDelay",
        "Distance",
        "Cancelled",
        "Diverted",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    if "Distance" in df.columns:
        df["Distance_km"] = df["Distance"] * 1.60934

    df = df.drop(columns=["Distance"])
    df = df.rename(columns={"Distance_km": "Distance"})
        
    return df


def build_airport_nodes(df):
    """
    Create airport-level node analysis table.

    Each row is one airport (IATA code), with counts and average metrics.
    """
    required_cols = ["Origin", "Dest"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    origin_stats = (
        df.groupby("Origin")
        .agg(
            origin_flight_count=("Origin", "size"),
            avg_actual_elapsed_time=("ActualElapsedTime", "mean"),
            avg_scheduled_elapsed_time=("CRSElapsedTime", "mean"),
            avg_air_time=("AirTime", "mean"),
            avg_distance=("Distance", "mean"),
            avg_dep_delay=("DepDelay", "mean"),
            cancelled_origin_count=("Cancelled", "sum"),
            diverted_origin_count=("Diverted", "sum"),
            unique_destinations=("Dest", "nunique"),
        )
        .reset_index()
        .rename(columns={"Origin": "IATA"})
    )

    dest_stats = (
        df.groupby("Dest")
        .agg(
            dest_flight_count=("Dest", "size"),
            avg_arr_delay=("ArrDelay", "mean"),
            unique_origins=("Origin", "nunique"),
        )
        .reset_index()
        .rename(columns={"Dest": "IATA"})
    )

    airport_nodes = pd.merge(origin_stats, dest_stats, on="IATA", how="outer")

    count_cols = [
        "origin_flight_count",
        "dest_flight_count",
        "cancelled_origin_count",
        "diverted_origin_count",
        "unique_destinations",
        "unique_origins",
    ]

    for col in count_cols:
        airport_nodes[col] = airport_nodes[col].fillna(0)

    airport_nodes["flight_count"] = (
        airport_nodes["origin_flight_count"] + airport_nodes["dest_flight_count"]
    )

    airport_nodes["unique_connections"] = (
        airport_nodes["unique_destinations"] + airport_nodes["unique_origins"]
    )

    airport_nodes["avg_flight_time"] = airport_nodes["avg_actual_elapsed_time"]

    round_cols = [
        "avg_flight_time",
        "avg_actual_elapsed_time",
        "avg_scheduled_elapsed_time",
        "avg_air_time",
        "avg_distance",
        "avg_dep_delay",
        "avg_arr_delay",
    ]

    for col in round_cols:
        airport_nodes[col] = airport_nodes[col].round(2)

    airport_nodes = airport_nodes[
        [
            "IATA",
            "flight_count",
            "origin_flight_count",
            "dest_flight_count",
            "avg_flight_time",
            "avg_actual_elapsed_time",
            "avg_scheduled_elapsed_time",
            "avg_air_time",
            "avg_distance",
            "avg_dep_delay",
            "avg_arr_delay",
            "cancelled_origin_count",
            "diverted_origin_count",
            "unique_destinations",
            "unique_origins",
            "unique_connections",
        ]
    ].sort_values("flight_count", ascending=False)

    return airport_nodes


def save_airport_nodes_for_period(files, period_name, output_dir="."):
    """
    Run the full pipeline for one period and save a CSV.

    Parameters
    ----------
    files : list of str
        CSV files to combine for the period.
    period_name : str
        Name of the period, used in the output filename.
    output_dir : str
        Directory to save the CSV.

    Returns
    -------
    pd.DataFrame
    """
    df = load_and_combine_csvs(files, period_name=period_name)
    df = clean_flight_data(df)
    airport_nodes = build_airport_nodes(df)

    safe_period_name = period_name.replace(" ", "_").replace("-", "_")
    output_path = os.path.join(output_dir, f"airport_nodes_{safe_period_name}.csv")

    airport_nodes.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

    return airport_nodes


In [14]:
period_1999_2000_files = ["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/1999.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2000.csv"]
period_2002_2003_files = ["/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2002.csv", "/Users/haseebshafi/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/4. Semester/02467 - ComSocSci/Project/2003.csv"]

nodes_1999_2000 = save_airport_nodes_for_period(
    period_1999_2000_files,
    "1999-2000"
)

nodes_2002_2003 = save_airport_nodes_for_period(
    period_2002_2003_files,
    "2002-2003"
)

print(nodes_1999_2000.head())
print(nodes_2002_2003.head())

Saved: ./airport_nodes_1999_2000.csv
Saved: ./airport_nodes_2002_2003.csv
    IATA  flight_count  origin_flight_count  dest_flight_count  \
150  ORD       1190170               595022             595148   
8    ATL       1058168               529573             528595   
51   DFW        984122               491949             492173   
107  LAX        807862               403880             403982   
156  PHX        733226               366629             366597   

     avg_flight_time  avg_actual_elapsed_time  avg_scheduled_elapsed_time  \
150           142.71                   142.71                      141.77   
8             119.88                   119.88                      120.38   
51            147.93                   147.93                      149.56   
107           162.65                   162.65                      163.38   
156           129.22                   129.22                      129.93   

     avg_air_time  avg_distance  avg_dep_delay  avg_arr_delay  \
1